# 31.07 - YOLO object detection fine-tuning

**Notebook type:** Solution notebook with theory, complete implementations, smoke checks, and test cases.

**Daily output:** Accuracy-first Torchvision detector + stratified OOF validation + fold ensemble.

The allowlist has no exact YOLO implementation. This notebook preserves YOLO label-format and detection concepts, but uses **Faster R-CNN ResNet50-FPN V2** as the executable accuracy-first Torchvision baseline. It is a two-stage detector, not YOLO. The model choice is based on official COCO weight metadata (46.7 box mAP), not a claim that it is universally best for every speed, memory, or dataset constraint.

## Core Ideas

- **YOLO versus Faster R-CNN:** YOLO is a one-stage family. Faster R-CNN first proposes regions and then classifies/refines them. The substitution is accuracy-oriented and must not be presented as exact YOLO practice.
- **Allowed model selection:** `FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT` is the official pretrained route. It may download a 167 MB checkpoint, so use it only when the weights are cached/attached and permitted by competition rules. The offline route constructs the same library architecture with no weights.
- **Custom Dataset boundary:** Competition data normally starts as files and metadata. The Dataset owns image loading, YOLO-to-pixel-box conversion, dtype checks, and optional unlabeled test rows.
- **Stratified K-fold:** Each image in this toy dataset has one class, so `StratifiedKFold` is valid. Real multi-object detection may need multilabel stratification, and patient/video/scene data needs group-aware splitting.
- **OOF evidence:** Every labeled observation is predicted by exactly one model that did not train on it. Aggregate metrics from the complete OOF table rather than choosing the most favorable fold.
- **Test ensemble:** Fold models predict the same test image. This compact notebook combines the winning class vote and score-weighted boxes. Production detection ensembles should use a carefully validated class-aware fusion method.

Official references:

- https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.detection.fasterrcnn_resnet50_fpn_v2.html
- https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html

The prepared fixture has four validation images per class in each fold, below the preferred ten-per-class target. It demonstrates the workflow mechanics; do not treat its metrics as a stable model ranking.

In [ ]:
import os
import csv
import time
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from PIL import Image, ImageDraw
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

SEED = 31
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_ROOT = "_day31_yolo_data"
IMAGE_DIR = os.path.join(DATA_ROOT, "images")
LABEL_DIR = os.path.join(DATA_ROOT, "labels")
TEST_IMAGE_DIR = os.path.join(DATA_ROOT, "test_images")
FOLD_DIR = os.path.join(DATA_ROOT, "folds")
for directory in [IMAGE_DIR, LABEL_DIR, TEST_IMAGE_DIR, FOLD_DIR]:
    os.makedirs(directory, exist_ok=True)
CLASS_NAMES = ["warm_square", "cool_square"]

## Prepared YOLO-Style Competition Data

The provided cell writes 24 labeled training images, YOLO text labels, a manifest, and four unlabeled test images. Fold membership is deliberately not stored here: the learner will create it deterministically from stable sample IDs and labels.

In [ ]:
for obsolete_name in ["dataset.yaml", "train.txt", "val.txt"]:
    obsolete_path = os.path.join(DATA_ROOT, obsolete_name)
    if os.path.isfile(obsolete_path):
        os.remove(obsolete_path)

manifest_rows = []
for class_id in range(2):
    for class_index in range(12):
        image = Image.new("RGB", (64, 64), color=(232, 235, 239))
        draw = ImageDraw.Draw(image)
        x1 = 7 + ((class_index * 7 + class_id * 5) % 29)
        y1 = 8 + ((class_index * 11 + class_id * 3) % 27)
        side = 17 + (class_index % 5)
        x2, y2 = x1 + side, y1 + side
        color = (220, 70, 65) if class_id == 0 else (55, 125, 225)
        draw.rectangle((x1, y1, x2, y2), fill=color, outline=(30, 30, 30), width=1)
        stem = f"class{class_id}_{class_index:02d}"
        image.save(os.path.join(IMAGE_DIR, stem + ".png"))
        xc, yc = (x1 + x2) / 128.0, (y1 + y2) / 128.0
        width, height = (x2 - x1) / 64.0, (y2 - y1) / 64.0
        with open(os.path.join(LABEL_DIR, stem + ".txt"), "w", encoding="utf-8") as handle:
            handle.write(f"{class_id} {xc:.6f} {yc:.6f} {width:.6f} {height:.6f}\n")
        manifest_rows.append({"stem": stem, "class_id": class_id})

with open(os.path.join(DATA_ROOT, "manifest.csv"), "w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["stem", "class_id"])
    writer.writeheader()
    writer.writerows(manifest_rows)

test_rows = []
for test_index in range(4):
    image = Image.new("RGB", (64, 64), color=(232, 235, 239))
    draw = ImageDraw.Draw(image)
    class_hint = test_index % 2
    x1, y1 = 10 + test_index * 6, 12 + test_index * 5
    side = 19
    color = (220, 70, 65) if class_hint == 0 else (55, 125, 225)
    draw.rectangle((x1, y1, x1 + side, y1 + side), fill=color, outline=(30, 30, 30), width=1)
    stem = f"test_{test_index:02d}"
    image.save(os.path.join(TEST_IMAGE_DIR, stem + ".png"))
    test_rows.append({"stem": stem})

print("labeled observations:", len(manifest_rows), "class support:", np.bincount([row["class_id"] for row in manifest_rows]).tolist())
print("unlabeled test observations:", len(test_rows))

## Exercise 31-A: Build a reusable custom detection Dataset

Do not wrap preloaded tensors in `TensorDataset` or use a bare `Subset`. Load each image from disk on demand. For labeled rows, convert one normalized YOLO `xywh` row to Torchvision pixel `xyxy` and shift class IDs from `{0,1}` to foreground IDs `{1,2}`. For test rows, omit targets.

**Return structure — `CompetitionDetectionDataset`:** A `Dataset` backed by file paths and stable row metadata. `dataset[i]` always returns `image` (CPU `torch.float32 [3,H,W]` in `[0,1]`) and `stem` (`str`). For labeled datasets it also returns `target`, a dictionary with `boxes` (CPU `torch.float32 [1,4]` pixel `xyxy`) and `labels` (CPU `torch.int64 [1]` in `{1,2}`), plus `class_id` (scalar CPU `torch.int64` in `{0,1}`). `select(indices)` returns a new `CompetitionDetectionDataset` over exactly those original row indices without using `torch.utils.data.Subset`.

In [ ]:
class CompetitionDetectionDataset(Dataset):
    def __init__(self, rows, image_dir, label_dir=None, indices=None):
        self.rows = list(rows)
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.indices = list(range(len(self.rows))) if indices is None else [int(index) for index in indices]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        row_index = self.indices[index]
        row = self.rows[row_index]
        stem = row["stem"]
        image_array = np.asarray(Image.open(os.path.join(self.image_dir, stem + ".png")).convert("RGB"), dtype=np.float32) / 255.0
        image = torch.from_numpy(image_array.transpose(2, 0, 1).copy())
        item = {"image": image, "stem": stem}
        if self.label_dir is not None:
            with open(os.path.join(self.label_dir, stem + ".txt"), "r", encoding="utf-8") as handle:
                parts = handle.readline().strip().split()
            if len(parts) != 5:
                raise ValueError(f"Expected one five-value YOLO row for {stem}")
            class_id = int(parts[0])
            xc, yc, width, height = [float(value) for value in parts[1:]]
            if class_id not in (0, 1) or any(value < 0.0 or value > 1.0 for value in [xc, yc, width, height]):
                raise ValueError(f"Invalid YOLO target for {stem}")
            height_px, width_px = image.shape[1:]
            boxes = torch.tensor([[(xc-width/2)*width_px, (yc-height/2)*height_px, (xc+width/2)*width_px, (yc+height/2)*height_px]], dtype=torch.float32)
            item["target"] = {"boxes": boxes, "labels": torch.tensor([class_id + 1], dtype=torch.int64)}
            item["class_id"] = torch.tensor(class_id, dtype=torch.int64)
        return item

    def select(self, indices):
        return CompetitionDetectionDataset(self.rows, self.image_dir, self.label_dir, indices=indices)


# Smoke check: load labeled and unlabeled files through the same custom Dataset type.
full_dataset = CompetitionDetectionDataset(manifest_rows, IMAGE_DIR, LABEL_DIR)
test_dataset = CompetitionDetectionDataset(test_rows, TEST_IMAGE_DIR)
print(full_dataset[0]["stem"], full_dataset[0]["target"])
print(test_dataset[0]["stem"], test_dataset[0]["image"].shape)

## Exercise 31-B: Create stable stratified folds

Use `StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)` with the one class label per image. Save fold-specific train and validation image lists so fold membership is auditable outside Python. Do not stratify ordinary multi-object data this way without defining an appropriate multilabel proxy.

**Return structure — `make_stratified_folds`:** A list of exactly `n_splits` dictionaries. Each has `fold` (`int`), `train_indices` (`list[int]`), `val_indices` (`list[int]`), `train_support` (`list[int]` length 2), and `val_support` (`list[int]` length 2). Across the list, every row index appears in exactly one validation list. The function also writes `fold_XX_train.txt` and `fold_XX_val.txt` under `fold_dir`, one relative image path per line.

In [ ]:
def make_stratified_folds(rows, n_splits=3, seed=SEED, fold_dir=FOLD_DIR):
    labels = np.asarray([int(row["class_id"]) for row in rows], dtype=np.int64)
    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    folds = []
    os.makedirs(fold_dir, exist_ok=True)
    for fold_number, (train_indices, val_indices) in enumerate(splitter.split(np.zeros(len(labels)), labels)):
        train_list = train_indices.astype(int).tolist()
        val_list = val_indices.astype(int).tolist()
        fold = {
            "fold": fold_number,
            "train_indices": train_list,
            "val_indices": val_list,
            "train_support": np.bincount(labels[train_indices], minlength=2).astype(int).tolist(),
            "val_support": np.bincount(labels[val_indices], minlength=2).astype(int).tolist(),
        }
        for split_name, indices in [("train", train_list), ("val", val_list)]:
            with open(os.path.join(fold_dir, f"fold_{fold_number:02d}_{split_name}.txt"), "w", encoding="utf-8") as handle:
                for index in indices:
                    handle.write(os.path.join("images", rows[index]["stem"] + ".png") + "\n")
        folds.append(fold)
    return folds


# Smoke check: print class support for every deterministic fold.
folds = make_stratified_folds(manifest_rows, n_splits=3)
print(pd.DataFrame([{key: fold[key] for key in ["fold", "train_support", "val_support"]} for fold in folds]).to_string(index=False))

## Exercise 31-C: Build the accuracy-first Torchvision detector

Use `fasterrcnn_resnet50_fpn_v2`. The offline route must disable detector and backbone weights. The pretrained route loads official COCO weights and replaces the ROI classifier with `FastRCNNPredictor` for three outputs including background. It may download weights and must remain opt-in.

**Return structure — `build_accuracy_first_detector`:** A CPU `torchvision.models.detection.FasterRCNN` module. In training mode it accepts an image list and target list and returns scalar loss tensors. In evaluation mode it returns `list[dict]` with `boxes` (`float32 [K,4]`), `scores` (`float32 [K]`), and `labels` (`int64 [K]`). No forward pass occurs inside this builder.

In [ ]:
def build_accuracy_first_detector(num_classes=3, use_pretrained=False, min_size=64, max_size=64):
    common = {
        "min_size": min_size, "max_size": max_size,
        "rpn_pre_nms_top_n_train": 200, "rpn_post_nms_top_n_train": 100,
        "rpn_pre_nms_top_n_test": 100, "rpn_post_nms_top_n_test": 50,
        "box_detections_per_img": 20,
    }
    if use_pretrained:
        model = fasterrcnn_resnet50_fpn_v2(weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT, **common)
        in_features = model.roi_heads.box_predictor.cls_score.in_features
        model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    else:
        model = fasterrcnn_resnet50_fpn_v2(weights=None, weights_backbone=None, num_classes=num_classes, **common)
    return model.cpu()


# Smoke check: construct the official architecture without a network download.
detector_smoke = build_accuracy_first_detector(use_pretrained=False)
print(type(detector_smoke).__name__, type(detector_smoke.roi_heads.box_predictor).__name__)

## Exercise 31-D: Train one complete fold

Create a fold-specific custom Dataset with `dataset.select`, train on every fold-training observation, and record native Faster R-CNN losses. The smoke check is intentionally one mini-batch; the next exercise performs complete training for every fold.

**Return structure — `train_fold_model`:** A tuple `(model, history)`. Position 0 is a trained `FasterRCNN` on `device`. Position 1 is a list of `epochs` dictionaries with `fold`, `epoch`, `observations` (`int`) and float losses `loss_classifier`, `loss_box_reg`, `loss_objectness`, `loss_rpn_box_reg`, and `total`. If `max_batches` is not `None`, observations reflect only that explicitly labeled structural run.

In [ ]:
def train_fold_model(dataset, fold, epochs=1, batch_size=4, learning_rate=0.0005, use_pretrained=False, device=DEVICE, max_batches=None):
    torch.manual_seed(SEED + int(fold["fold"]))
    train_dataset = dataset.select(fold["train_indices"])
    loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, generator=torch.Generator().manual_seed(SEED + int(fold["fold"])), collate_fn=lambda items: items)
    model = build_accuracy_first_detector(use_pretrained=use_pretrained).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)
    history = []
    loss_names = ["loss_classifier", "loss_box_reg", "loss_objectness", "loss_rpn_box_reg"]
    for epoch in range(1, epochs + 1):
        model.train()
        totals = {name: 0.0 for name in loss_names + ["total"]}
        observations = 0
        for batch_number, batch in enumerate(loader):
            images = [item["image"].to(device) for item in batch]
            targets = [{key: value.to(device) for key, value in item["target"].items()} for item in batch]
            optimizer.zero_grad()
            losses = model(images, targets)
            total = sum(losses.values())
            total.backward()
            optimizer.step()
            count = len(batch)
            observations += count
            for name in loss_names:
                totals[name] += float(losses[name].detach().cpu()) * count
            totals["total"] += float(total.detach().cpu()) * count
            if max_batches is not None and batch_number + 1 >= max_batches:
                break
        history.append({"fold": int(fold["fold"]), "epoch": epoch, "observations": observations, **{name: totals[name] / observations for name in totals}})
    return model, history


# Smoke check: one speed-only structural batch; full-fold training occurs next.
structural_model, structural_history = train_fold_model(full_dataset, folds[0], max_batches=1)
print("structural fold record:", structural_history[-1])

## Exercise 31-E: Train all folds and create complete OOF evidence

Train one fresh model per fold. Predict only that fold's validation indices, retain the top detection, and mark a correct detection only when class matches and IoU reaches the threshold. Print fold support and metrics plus the aggregate OOF result. A random-weight one-epoch run proves mechanics, not competitiveness.

**Return structure — `run_stratified_cv`:** A dictionary with `models`, a list of one trained `FasterRCNN` per fold; `histories`, a list of fold history lists; `fold_table`, a `pandas.DataFrame` with one row per fold and columns `fold`, `train_size`, `val_size`, `val_support`, `detection_accuracy`, and `runtime_seconds`; and `oof_rows`, a `list[dict]` of length `len(dataset)`. Every OOF row has `row_index`, `stem`, `fold`, `truth`, `predicted`, `confidence`, `iou`, and `correct`.

In [ ]:
def run_stratified_cv(dataset, folds, epochs=1, batch_size=4, use_pretrained=False, confidence_threshold=0.2, iou_threshold=0.5, device=DEVICE):
    models, histories, fold_records, oof_rows = [], [], [], []
    for fold in folds:
        start = time.perf_counter()
        model, history = train_fold_model(dataset, fold, epochs=epochs, batch_size=batch_size, use_pretrained=use_pretrained, device=device)
        models.append(model)
        histories.append(history)
        correct_count = 0
        model.eval()
        with torch.no_grad():
            for row_index in fold["val_indices"]:
                item = dataset[dataset.indices.index(row_index)]
                output = model([item["image"].to(device)])[0]
                truth = int(item["class_id"])
                if len(output["scores"]) == 0 or float(output["scores"][0]) < confidence_threshold:
                    predicted, confidence, iou = -1, 0.0, 0.0
                else:
                    predicted = int(output["labels"][0]) - 1
                    confidence = float(output["scores"][0])
                    box, target = output["boxes"][0].cpu(), item["target"]["boxes"][0]
                    top_left, bottom_right = torch.maximum(box[:2], target[:2]), torch.minimum(box[2:], target[2:])
                    intersection_wh = (bottom_right - top_left).clamp(min=0)
                    intersection = float(intersection_wh[0] * intersection_wh[1])
                    box_area = float((box[2]-box[0]).clamp(min=0) * (box[3]-box[1]).clamp(min=0))
                    target_area = float((target[2]-target[0]) * (target[3]-target[1]))
                    iou = intersection / max(box_area + target_area - intersection, 1e-7)
                correct = bool(predicted == truth and iou >= iou_threshold)
                correct_count += int(correct)
                oof_rows.append({"row_index": row_index, "stem": item["stem"], "fold": int(fold["fold"]), "truth": truth, "predicted": predicted, "confidence": confidence, "iou": float(iou), "correct": correct})
        fold_records.append({"fold": int(fold["fold"]), "train_size": len(fold["train_indices"]), "val_size": len(fold["val_indices"]), "val_support": fold["val_support"], "detection_accuracy": correct_count / len(fold["val_indices"]), "runtime_seconds": time.perf_counter() - start})
    oof_rows.sort(key=lambda row: row["row_index"])
    return {"models": models, "histories": histories, "fold_table": pd.DataFrame(fold_records), "oof_rows": oof_rows}


# Smoke check: full three-fold training and complete OOF evidence.
cv_result = run_stratified_cv(full_dataset, folds, epochs=1, batch_size=4, use_pretrained=False)
print(cv_result["fold_table"].to_string(index=False))
print("OOF detection accuracy:", np.mean([row["correct"] for row in cv_result["oof_rows"]]))

## Exercise 31-F: Ensemble fold models on unlabeled test images

Collect each fold model's top detection. Sum scores as class votes, choose the winning class, then calculate a score-weighted mean box among models voting for that class. This compact fusion is transparent; for real multi-object detection, validate class-aware NMS or weighted box fusion across all candidates.

**Return structure — `ensemble_fold_models`:** A list of length `len(test_dataset)`. Each dictionary has `stem` (`str`), `predicted` (`int` zero-based or `-1`), `confidence` (`float`), `box_xyxy` (`list[float]` length 4 or empty list), and `model_count` (`int`, exactly the number of supplied models). One dictionary is returned even when no model clears the threshold.

In [ ]:
def ensemble_fold_models(models, test_dataset, confidence_threshold=0.2, device=DEVICE):
    for model in models:
        model.to(device).eval()
    predictions = []
    with torch.no_grad():
        for item in test_dataset:
            candidates = []
            for model in models:
                output = model([item["image"].to(device)])[0]
                if len(output["scores"]) and float(output["scores"][0]) >= confidence_threshold:
                    candidates.append({"class_id": int(output["labels"][0]) - 1, "score": float(output["scores"][0]), "box": output["boxes"][0].cpu().numpy()})
            if not candidates:
                predictions.append({"stem": item["stem"], "predicted": -1, "confidence": 0.0, "box_xyxy": [], "model_count": len(models)})
                continue
            class_scores = {class_id: sum(candidate["score"] for candidate in candidates if candidate["class_id"] == class_id) for class_id in [0, 1]}
            winner = max(class_scores, key=class_scores.get)
            winners = [candidate for candidate in candidates if candidate["class_id"] == winner]
            weights = np.asarray([candidate["score"] for candidate in winners], dtype=np.float32)
            boxes = np.stack([candidate["box"] for candidate in winners])
            fused_box = np.average(boxes, axis=0, weights=weights).astype(float).tolist()
            predictions.append({"stem": item["stem"], "predicted": int(winner), "confidence": float(np.mean(weights)), "box_xyxy": fused_box, "model_count": len(models)})
    return predictions


# Smoke check: fold-model ensemble on every unlabeled test image.
test_predictions = ensemble_fold_models(cv_result["models"], test_dataset)
print(pd.DataFrame(test_predictions).to_string(index=False))

## Test Cases

Tests enforce custom Dataset behavior, disjoint stratified folds, complete OOF coverage, fold-specific models, and test ensemble schema. They do not require a random-weight one-epoch detector to achieve a target score.

**Return structure — `run_day31_tests`:** Returns `None`. Success is communicated by assertions completing and the exact printed message `Day 31 tests passed`.

In [ ]:
def run_day31_tests():
    assert os.path.isfile(os.path.join(DATA_ROOT, "manifest.csv"))
    assert len(full_dataset) == len(manifest_rows) == 24 and len(test_dataset) == 4
    labeled = full_dataset[0]
    assert set(labeled) == {"image", "stem", "target", "class_id"}
    assert labeled["image"].shape == (3, 64, 64) and labeled["image"].dtype == torch.float32
    assert labeled["target"]["boxes"].shape == (1, 4) and labeled["target"]["boxes"].dtype == torch.float32
    assert labeled["target"]["labels"].dtype == torch.int64 and int(labeled["target"]["labels"][0]) in (1, 2)
    assert set(test_dataset[0]) == {"image", "stem"}
    all_indices = set(range(len(manifest_rows)))
    validation_indices = []
    for fold in folds:
        train_set, val_set = set(fold["train_indices"]), set(fold["val_indices"])
        assert train_set.isdisjoint(val_set) and train_set | val_set == all_indices
        assert fold["train_support"] == [8, 8] and fold["val_support"] == [4, 4]
        validation_indices.extend(fold["val_indices"])
        assert os.path.isfile(os.path.join(FOLD_DIR, f"fold_{fold['fold']:02d}_train.txt"))
        assert os.path.isfile(os.path.join(FOLD_DIR, f"fold_{fold['fold']:02d}_val.txt"))
    assert sorted(validation_indices) == list(range(24)) and len(set(validation_indices)) == 24
    assert type(cv_result["models"][0]).__name__ == "FasterRCNN" and len(cv_result["models"]) == 3
    assert len(cv_result["histories"]) == 3 and cv_result["fold_table"].shape[0] == 3
    assert len(cv_result["oof_rows"]) == 24
    assert [row["row_index"] for row in cv_result["oof_rows"]] == list(range(24))
    assert len(test_predictions) == len(test_dataset)
    assert set(test_predictions[0]) == {"stem", "predicted", "confidence", "box_xyxy", "model_count"}
    assert all(row["model_count"] == 3 and len(row["box_xyxy"]) in (0, 4) for row in test_predictions)
    print("Day 31 tests passed")


run_day31_tests()

## Day 31 Checklist

- [ ] I can explain why Faster R-CNN V2 is an accuracy-first substitute, not YOLO and not universally best.
- [ ] My custom Dataset owns file loading and YOLO-to-Torchvision target conversion.
- [ ] I created deterministic stratified folds without `Subset`, `TensorDataset`, or `train_test_split`.
- [ ] Every labeled observation has exactly one out-of-fold prediction.
- [ ] I inspected fold support, fold metrics, and aggregate OOF evidence.
- [ ] I ensembled all fold models on unlabeled test images.
- [ ] I know when detection needs multilabel or group-aware splitting instead of plain `StratifiedKFold`.
- [ ] I will use pretrained weights only when cached/attached and competition-legal.